# Bayesian Epistemic Routing — label questions with Qwen2.5-7B-Instruct on Kaggle (free)

**Why this model:** one of the most widely used open 7B instruct models (Apache-2.0, no licence gate), standard
Hugging Face architecture (no custom code, no flash-attn), and common in RAG / hallucination papers.
It runs in plain **16-bit** split over Kaggle's two T4s, so no 4-bit quantisation noise in the labels.

**Setup (once)**
1. Kaggle account → *Settings* → verify your phone number (free, no card).
2. Hugging Face (free): *Settings → Access Tokens → Create new token*, type **Write**. In this Kaggle notebook:
   *Add-ons → Secrets → Add secret*, name `HF_TOKEN`, paste the token, tick it for this notebook.
   → labels are backed up every 50 questions to a **private** HF dataset `<your username>/prob-reas-labels`.
3. On your Mac, make the upload zip:
   ```
   cd ~/Documents/prob_reas
   zip -r ~/Desktop/prob_reas_kaggle.zip *.py requirements*.txt kb labels -x "kb/wiki_cache.jsonl"
   ```
4. Kaggle → *Datasets* → `prob-reas` → *New Version* → upload the zip. Import this notebook; *Add Input* → `prob-reas`;
   **Accelerator: GPU T4 x2**; **Internet: on**.

**Running so nothing is lost, even with your laptop off**
* Click **Save Version → Save & Run All (Commit)**. A commit runs on Kaggle's side (up to 12 h) with the browser
  closed — no idle timeout. This is the only way a run keeps going while you are offline: an *interactive*
  session is always stopped some time after the browser goes idle, and nothing inside the notebook can prevent that.
* Whatever happens (timeout, crash, quota), at most 50 questions are lost: every 50 the file is uploaded to your
  private HF dataset, and the next run downloads it and **continues** from there. Just commit again.
* When it finishes: version → **Output** tab → `labels.zip`, or download the `.jsonl` directly from the HF dataset page.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Copy the uploaded project into a writable folder
import os, glob, shutil
hits = glob.glob('/kaggle/input/**/label_dataset.py', recursive=True)
assert hits, 'Add your prob-reas dataset as an input (right panel -> Add Input)'
shutil.copytree(os.path.dirname(hits[0]), '/kaggle/working/project', dirs_exist_ok=True)
os.chdir('/kaggle/working/project')
os.environ['HF_HOME'] = '/tmp/hf'            # 15 GB of weights: keep them out of /kaggle/working (20 GB cap)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('project files:', sorted(os.listdir('.'))[:25])

In [ ]:
# Hugging Face token from Kaggle Secrets (never paste it into code) -> off-machine checkpoints every 50 questions.
# The run REFUSES to start without it, so a dead session can never lose the work again.
import os
from kaggle_secrets import UserSecretsClient
try:
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    raise SystemExit('No HF_TOKEN secret attached: Add-ons -> Secrets -> HF_TOKEN (Write token), tick it for this notebook') from e
from huggingface_hub import HfApi
print('HF_TOKEN OK, logged in as', HfApi().whoami()['name'])
assert os.path.exists('/kaggle/working/project/checkpoint.py'), 'old upload: re-zip the project (checkpoint.py is missing) and add a new dataset version'
CHECKPOINT = '--checkpoint auto --checkpoint-every 50'

In [ ]:
# Libraries (Kaggle's own torch / transformers are used as-is)
!pip install -q spacy click pgmpy faiss-cpu accelerate 2>&1 | grep -v "already satisfied" | tail -3
!python -m spacy download en_core_web_sm -q 2>&1 | tail -2
import torch, transformers, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| transformers', transformers.__version__,
      '| GPUs', torch.cuda.device_count())

In [ ]:
# 1. LOAD + SMOKE TEST
LLM = 'qwen2.5-7b'          # 16-bit over both T4s.  Alternatives: 'llama3.1-8b' (accept licence on HF first), 'qwen2.5-7b-nf4' (one T4)
import time
from llm_backends import make_backend, build_prompt
t0 = time.time()
llm = make_backend(LLM)
print(f'loaded {llm.name} in {time.time() - t0:.0f}s on', {str(p.device) for p in llm.model.parameters()})
for q, ctx in [("What is the capital of France?", "Paris is the capital and largest city of France."),
               ("Who was the composer of Dream On (An Indian Lullaby)?", "Victor Herbert composed the song Dream On (An Indian Lullaby) in 1912."),
               ("Who was the screenwriter for Wit?", "Emma Thompson wrote the screenplay for the 2001 film Wit.")]:
    t0 = time.time()
    print(repr(llm.generate(build_prompt(q, [ctx]))), f'in {time.time() - t0:.1f}s')

In [ ]:
# 2. QUALITY CHECK (already passed - set RUN_CHECK = True to repeat it)
RUN_CHECK = False
if RUN_CHECK:
    # 2. QUALITY CHECK before labelling anything (~3 min)
    #  a) the 18 questions StripedHyena garbled / got right, with the same retrieved passages
    #  b) how well the model predicts plain Wikipedia text by position (healthy = low and flat/falling)
    import glob
    from diagnose_model import question_prompts, answer_all, health_texts, health, print_health
    old = glob.glob('labels/*StripedHyena*noflash.jsonl')
    assert old, 'zip the labels/ folder with the code (see step 2 at the top)'
    sample, prompts = question_prompts(old[0], 'kb')
    replies, dmg = answer_all(llm, sample, prompts, llm.name.split('/')[-1][:16])
    for r, a in zip(sample, replies):
        print(f"   {r['question'][:45]:<46} gold {r['answers'][0][:18]:<19} -> {a[:40]!r}")
    print_health('wiki loss', health(llm, health_texts('kb', llm.tokenizer)))
    print('\nGo ahead with labelling if: damaged <= 2/18 and wiki ppl roughly < 15 at every position.')

In [ ]:
# 3. LABEL ALL QUESTIONS. Saves after every question -> /kaggle/working/labels/ ; resumes from the HF backup or an uploaded partial file
import os, gc, glob, shutil, torch
os.chdir('/kaggle/working/project')
os.makedirs('/kaggle/working/labels', exist_ok=True)
for f in glob.glob('/kaggle/working/project/labels/Qwen_*.jsonl'):          # partial run uploaded with the dataset
    dst = os.path.join('/kaggle/working/labels', os.path.basename(f))
    if not os.path.exists(dst):
        shutil.copy(f, dst); print('resuming from', f, sum(1 for _ in open(f)), 'rows')
globals().pop('llm', None); gc.collect(); torch.cuda.empty_cache()     # free the GPUs: the script loads its own copy
!python /kaggle/working/project/label_dataset.py --llm {LLM} --limit 0 --out /kaggle/working/labels {CHECKPOINT}

In [ ]:
# 4. Zip the labels for download (Output tab -> labels.zip)
import os, shutil
from IPython.display import FileLink
shutil.make_archive('/kaggle/working/labels', 'zip', '/kaggle/working/labels')
os.chdir('/kaggle/working')
FileLink('labels.zip')

**After the run:** unzip `labels.zip` into `prob_reas/labels/` on your Mac, then re-grade with the improved grader (no GPU needed):
```
python regrade_labels.py labels/Qwen_Qwen2.5-7B-Instruct-bfloat16.jsonl
```